# Paso 4 — Clustering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import Normalizer
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from IPython.display import display
import umap.umap_ as umap
import hdbscan
import itertools
from tqdm import tqdm 
import time
import sys


## NORMALIZACIÓN

In [ ]:
# --- Rutas y Configuración ---
BASE_OUTPUT_DIR = './output' 
INPUT_EMBEDDINGS_FILE = './output/embeddings.csv'
OUTPUT_NORMALIZED_FILE = './output/normalized_embeddings.csv'

print(f"Cargando embeddings ORIGINALES desde: {INPUT_EMBEDDINGS_FILE}")
df_original = pd.read_csv(INPUT_EMBEDDINGS_FILE)
display(df_original.head())

# --- Extracción de IDs y embeddings ---
 # Asume que la primera columna es el ID y el resto son las features
protein_ids = df_original.iloc[:, 0]
embedding_columns = df_original.columns[1:]
embeddings_original = df_original[embedding_columns].values

print(f"\nForma de embeddings: {embeddings_original.shape}")
print(f"Tipo de datos: {embeddings_original.dtype}")

In [ ]:
# --- Revisión de calidad ---
print("\n--- Verificación de calidad de los embeddings ---")
num_nan = np.isnan(embeddings_original).any(axis=1).sum()
num_inf = np.isinf(embeddings_original).any(axis=1).sum()
num_zeros = np.all(embeddings_original == 0, axis=1).sum()

print(f"Número de filas con NaN: {num_nan}")
print(f"Número de filas con Inf: {num_inf}")
print(f"Número de vectores de ceros: {num_zeros}")

# Normas L2 antes de normalización
l2_norms_original = np.linalg.norm(embeddings_original, axis=1)
print(f"Mínima norma L2: {np.min(l2_norms_original):.5f}")
print(f"Máxima norma L2: {np.max(l2_norms_original):.5f}")
print(f"Promedio norma L2: {np.mean(l2_norms_original):.5f}")


In [ ]:
print("\n--- Normalizando embeddings con sklearn Normalizer (L2) ---")

# Creamos el normalizador L2
l2_normalizer = Normalizer(norm='l2')

# Aplicamos la normalización a todos los vectores fila
embeddings_normalized = l2_normalizer.fit_transform(embeddings_original)

# Verificamos normas L2 post-normalización
l2_post = np.linalg.norm(embeddings_normalized, axis=1)

print(f"Norma L2 mínima después: {np.min(l2_post):.10f}")
print(f"Norma L2 máxima después: {np.max(l2_post):.10f}")
print(f"Norma L2 media después: {np.mean(l2_post):.10f}")
print(f"¿Todas las normas son ≈1? {np.allclose(l2_post, 1.0, atol=1e-7)}")

  
df_normalized = pd.DataFrame(embeddings_normalized, columns=embedding_columns)
df_normalized.insert(0, df_original.columns[0], protein_ids)

print(f"\nGuardando embeddings normalizados en: {OUTPUT_NORMALIZED_FILE}")
df_normalized.to_csv(OUTPUT_NORMALIZED_FILE, index=False)

# Estadísticas descriptivas
print("\n--- Estadísticas de los embeddings normalizados (Primeras 5 dim) ---")
display(df_normalized[embedding_columns].iloc[:, :5].describe())

## KMEANS

In [ ]:
# --- Configuración de Rutas ---
BASE_OUTPUT_DIR = './output' 
EMBEDDINGS_PATH = os.path.join(BASE_OUTPUT_DIR, "normalized_embeddings.csv")

# Directorio para guardar los gráficos de resultados de optimización
RESULTS_GRAPH_DIR = os.path.join(BASE_OUTPUT_DIR, "clustering_optimization_results")
os.makedirs(RESULTS_GRAPH_DIR, exist_ok=True)
print(f"Directorio de resultados creado/verificado: {RESULTS_GRAPH_DIR}")

# --- Carga de Embeddings Normalizados ---
print(f"\nCargando embeddings normalizados desde: {EMBEDDINGS_PATH}")
try:
    df_normalized_embeddings = pd.read_csv(EMBEDDINGS_PATH)
    print("Embeddings normalizados cargados exitosamente.")
except FileNotFoundError:
    print(f"🚨 ERROR: No se encontró el archivo de embeddings normalizados en '{EMBEDDINGS_PATH}'.")
    print("Asegúrate de ejecutar y guardar la celda de normalización primero.")
    df_normalized_embeddings = pd.DataFrame() 

if not df_normalized_embeddings.empty:
    protein_ids = df_normalized_embeddings.iloc[:, 0]
    embedding_columns = df_normalized_embeddings.columns[1:]
    embeddings_data = df_normalized_embeddings[embedding_columns].values

    print(f"Forma de los embeddings cargados: {embeddings_data.shape}")

    # --- Verificación de Normalización ---
    l2_norms_loaded = np.linalg.norm(embeddings_data, axis=1)
    print(f"\n--- Verificación de Normas L2 de los Embeddings ---")
    print(f"Mínima norma L2: {np.min(l2_norms_loaded):.10f}")
    if np.isclose(np.mean(l2_norms_loaded), 1.0, atol=1e-5):
        print("✔️ Las normas L2 están correctamente normalizadas a ~1.0.")
    else:
        print("⚠️ ADVERTENCIA: Las normas L2 no están centradas en 1.0. Revisa el paso anterior.")
else:
    embeddings_data = None
    print("⚠️ Deteniendo K-Means: No hay datos de embeddings disponibles.")

In [ ]:
if embeddings_data is None:
    print("🚨 ERROR: No se puede ejecutar K-Means, 'embeddings_data' no está cargado. Ejecute la Celda 1.")
else:
    print("\n--- BÚSQUEDA DEL MEJOR K PARA K-MEANS ---")
    
    # Definir el rango de K a probar (máximo 50 o número de muestras - 1)
    max_k_kmeans = min(50, embeddings_data.shape[0] - 1)
    if max_k_kmeans < 2:
        print("ERROR: No hay suficientes muestras para evaluar K-Means con al menos 2 clusters.")
    else:
        k_range = range(2, max_k_kmeans + 1)
        kmeans_results = []
        
        for k in k_range:
            print(f"Probando K-Means con K = {k}...")
            # n_init='auto' es el valor predeterminado para sklearn > 1.2
            kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto', verbose=0)
            labels = kmeans.fit_predict(embeddings_data)
            inertia = kmeans.inertia_

            # Calcular métricas solo si hay al menos 2 clusters únicos
            num_unique_clusters = len(np.unique(labels))
            if num_unique_clusters >= 2 and embeddings_data.shape[0] > 1:
                silhouette = silhouette_score(embeddings_data, labels)
                davies_bouldin = davies_bouldin_score(embeddings_data, labels)
                calinski_harabasz = calinski_harabasz_score(embeddings_data, labels)
            else:
                silhouette, davies_bouldin, calinski_harabasz = np.nan, np.nan, np.nan

            kmeans_results.append({
                'K': k,
                'Inertia': inertia,
                'Silhouette Score': silhouette,
                'Davies-Bouldin Index': davies_bouldin,
                'Calinski-Harabasz Index': calinski_harabasz
            })

        kmeans_df = pd.DataFrame(kmeans_results)
        print("✔️ Búsqueda de K-Means completada.")
        print("\n--- Tabla de Resultados de K-Means por K ---")
        print(kmeans_df.to_string(index=False))

In [ ]:
if 'kmeans_df' not in locals():
    print("🚨 ERROR: No se puede generar el reporte, 'kmeans_df' no existe. Ejecute la Celda 2.")
else:
    print("\n--- Tabla de Resultados de K-Means por K ---")
    display(kmeans_df)

    # --- Generación de Gráficos de Optimización ---
    plt.figure(figsize=(15, 12))
    k_unique = list(kmeans_df['K'].unique())

    # 1. Método del Codo (Elbow Method)
    plt.subplot(2, 2, 1)
    plt.plot(kmeans_df['K'], kmeans_df['Inertia'], marker='o', linestyle='-')
    plt.title('K-Means: Método del Codo (Inercia)', fontsize=14)
    plt.xlabel('Número de Clusters (K)', fontsize=12)
    plt.ylabel('Inercia', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.xticks(k_unique)

    # 2. Silhouette Score (Maximizar)
    plt.subplot(2, 2, 2)
    plt.plot(kmeans_df['K'], kmeans_df['Silhouette Score'], marker='o', linestyle='-', color='green')
    plt.title('K-Means: Silhouette Score (Maximizar)', fontsize=14)
    plt.xlabel('Número de Clusters (K)', fontsize=12)
    plt.ylabel('Puntuación Silhouette', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.xticks(k_unique)
    plt.axhline(y=0.0, color='red', linestyle='--', linewidth=1)

    # 3. Davies-Bouldin Index (Minimizar)
    plt.subplot(2, 2, 3)
    plt.plot(kmeans_df['K'], kmeans_df['Davies-Bouldin Index'], marker='o', linestyle='-', color='red')
    plt.title('K-Means: Davies-Bouldin Index (Minimizar)', fontsize=14)
    plt.xlabel('Número de Clusters (K)', fontsize=12)
    plt.ylabel('Índice Davies-Bouldin', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.xticks(k_unique)

    # 4. Calinski-Harabasz Index (Maximizar)
    plt.subplot(2, 2, 4)
    plt.plot(kmeans_df['K'], kmeans_df['Calinski-Harabasz Index'], marker='o', linestyle='-', color='purple')
    plt.title('K-Means: Calinski-Harabasz Index (Maximizar)', fontsize=14)
    plt.xlabel('Número de Clusters (K)', fontsize=12)
    plt.ylabel('Índice Calinski-Harabasz', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.xticks(k_unique)

    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
    plt.suptitle('Evaluación de Métricas de K-Means por K', y=0.99, fontsize=16)
    
    # Guardar gráfico
    graph_save_path = os.path.join(RESULTS_GRAPH_DIR, "kmeans_k_evaluation_metrics.png")
    plt.savefig(graph_save_path, dpi=300)
    print(f"\n✅ Gráficos de optimización de K-Means guardados en: {graph_save_path}")
    plt.show()

    # --- Determinación del Mejor K ---
    print("\n--- Determinación del Mejor K para K-Means Sugerido por Cada Métrica ---")
    kmeans_filtered = kmeans_df.dropna()
    
    if not kmeans_filtered.empty:
        # Encontrar los K óptimos
        best_k_silhouette = kmeans_filtered.loc[kmeans_filtered['Silhouette Score'].idxmax(), 'K']
        best_k_calinski_harabasz = kmeans_filtered.loc[kmeans_filtered['Calinski-Harabasz Index'].idxmax(), 'K']
        best_k_davies_bouldin = kmeans_filtered.loc[kmeans_filtered['Davies-Bouldin Index'].idxmin(), 'K']
        
        print(f"K-Means | Mejor K por Silhouette Score (más alto): {best_k_silhouette}")
        print(f"K-Means | Mejor K por Davies-Bouldin Index (más bajo): {best_k_davies_bouldin}")
        print(f"K-Means | Mejor K por Calinski-Harabasz Index (más alto): {best_k_calinski_harabasz}")
        
        # Sugerencia final (generalmente se prioriza Silhouette)
        optimal_k_kmeans_suggested = best_k_silhouette
        print(f"\nK-Means | K sugerido para usar (basado en Silhouette): {optimal_k_kmeans_suggested}")
    else:
        print("\nK-Means | No se pudo sugerir un K óptimo automáticamente.")

    print("\n✔️ Análisis de optimización de KMEANS completado.")

In [ ]:
if embeddings_data is None or optimal_k_kmeans_suggested is None:
    print("🚨 ERROR: Datos de embeddings o K óptimo no disponibles.")
else:
    
    print("\n--- 1. Reducción de Dimensionalidad con UMAP ---")
    
    # UMAP es sensible a la escala, aunque los datos ya estén normalizados L2.
    # UMAP prefiere datos densos, los embeddings lo son.
    # Ajustar n_neighbors y min_dist puede mejorar la visualización.
    reducer = umap.UMAP(
        n_neighbors=15, 
        min_dist=0.1, 
        n_components=2, 
        metric='cosine', # Usamos coseno ya que los embeddings son L2 normalizados
        random_state=42
    )
    
    # Reducción de dimensionalidad
    embeddings_2d = reducer.fit_transform(embeddings_data)
    print(f"Embeddings reducidos a 2D. Forma: {embeddings_2d.shape}")
    

    print("\n K-Means Óptimo y Visualización UMAP ---")

    # Ejecutar K-Means con el K óptimo sugerido
    k_opt = int(optimal_k_kmeans_suggested)
    kmeans_opt = KMeans(n_clusters=k_opt, random_state=42, n_init='auto', verbose=0)
    kmeans_labels = kmeans_opt.fit_predict(embeddings_data)
    
    plt.figure(figsize=(10, 8))
    sns.scatterplot(
        x=embeddings_2d[:, 0], 
        y=embeddings_2d[:, 1], 
        hue=kmeans_labels, 
        palette=sns.color_palette("hsv", k_opt), 
        legend='full', 
        alpha=0.6,
        s=15
    )
    plt.title(f'K-Means (K={k_opt}) sobre Embeddings UMAP (Métrica Coseno)', fontsize=16)
    plt.xlabel('UMAP 1', fontsize=12)
    plt.ylabel('UMAP 2', fontsize=12)
    plt.grid(True, alpha=0.3)
    
    # Guardar gráfico
    umap_kmeans_path = os.path.join(RESULTS_GRAPH_DIR, f"umap_kmeans_k_{k_opt}_plot.png")
    plt.savefig(umap_kmeans_path, dpi=300)
    print(f"Gráfico K-Means/UMAP guardado en: {umap_kmeans_path}")
    plt.show()

## DBSCAN

In [ ]:
if 'embeddings_data' not in locals() or embeddings_data is None:
    print("🚨 ERROR: No se puede ejecutar DBSCAN. 'embeddings_data' no está cargado. Ejecute la Celda 1.")
else:
    import itertools
    from sklearn.cluster import DBSCAN
    from scipy.spatial.distance import cosine # Necesario para metric='cosine'
    import seaborn as sns # Para mapas de calor
    
    # Usamos X como el alias para los embeddings
    X = embeddings_data 
    print(f"Dimensiones de los embeddings (X): {X.shape}")

    # --- Definición de Rangos de Búsqueda (Coseno) ---
    
    # Rango de eps: Distancia Coseno (de 0 a 1 o 2 en teoría, pero más efectivo cerca de 0-0.5)
    eps_range = np.concatenate((
        np.linspace(0.001, 0.05, 10), # Valores muy pequeños
        np.linspace(0.05, 0.2, 10),  # Rango de interés típico
        np.linspace(0.2, 0.8, 10),   # Rango medio
        np.linspace(0.8, 1.5, 5)     # Valores más grandes
    ))
    eps_range = np.sort(np.unique(np.round(eps_range, 4)))
    print(f"\nRango de eps (Coseno): [{eps_range.min():.4f}, {eps_range.max():.4f}] con {len(eps_range)} puntos.")


    # Rango de min_samples: Mínimo número de puntos para formar un cluster
    min_samples_range = np.unique(np.sort(np.concatenate((
        np.arange(2, 20, 2),  
        np.arange(20, 101, 10), 
        np.arange(100, 201, 50) 
    ))))
    print(f"Rango de min_samples: {min_samples_range[0]}-{min_samples_range[-1]} con {len(min_samples_range)} puntos.")

    # Inicializar lista de resultados
    dbscan_results = []
    total_combinations = len(eps_range) * len(min_samples_range)
    print(f"Combinaciones totales a probar: {total_combinations}")

In [ ]:
if 'X' not in locals():
    print("🚨 ERROR: No se puede ejecutar la búsqueda.")
else:
    print("\n--- BÚSQUEDA DE PARÁMETROS PARA DBSCAN ---")
    
    current_combination = 0
    
    for eps_val in eps_range:
        for min_s_val in min_samples_range:
            current_combination += 1
            print(f"Probando DBSCAN con eps={eps_val:.4f}, min_samples={min_s_val} ({current_combination}/{total_combinations})...")
            
            # Ejecutar DBSCAN con Distancia Coseno
            dbscan = DBSCAN(eps=eps_val, min_samples=min_s_val, metric='cosine')
            clusters = dbscan.fit_predict(X)

            # --- Análisis de Resultados ---
            n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
            n_noise = list(clusters).count(-1) 
            total_points = len(clusters)
            noise_percentage = (n_noise / total_points) * 100 if total_points > 0 else 0

            silhouette, davies_bouldin, calinski_harabasz = np.nan, np.nan, np.nan

            # Calcular métricas solo si hay al menos 2 clusters válidos y suficientes puntos
            if n_clusters >= 2:
                labels_filtered = clusters[clusters != -1]
                data_filtered = X[clusters != -1]

                if len(set(labels_filtered)) >= 2 and len(labels_filtered) >= 2:
                    try:
                        silhouette = silhouette_score(data_filtered, labels_filtered, metric='cosine')
                        # Nota: Davies-Bouldin y Calinski-Harabasz usan distancia Euclidiana por defecto,
                        # pero pueden aplicarse a embeddings L2 normalizados.
                        davies_bouldin = davies_bouldin_score(data_filtered, labels_filtered)
                        calinski_harabasz = calinski_harabasz_score(data_filtered, labels_filtered)
                    except ValueError as e:
                        # Esto puede ocurrir si un cluster tiene solo un punto.
                        # print(f"Error en métricas: {e}")
                        pass
                
            dbscan_results.append({
                'eps': eps_val,
                'min_samples': min_s_val,
                'Num Clusters (valid)': n_clusters,
                'Num Ruido': n_noise,
                'Porcentaje Ruido': noise_percentage,
                'Silhouette Score': silhouette,
                'Davies-Bouldin Index': davies_bouldin,
                'Calinski-Harabasz Index': calinski_harabasz
            })

    dbscan_df = pd.DataFrame(dbscan_results)
    print("✔️ Búsqueda de DBSCAN completada.")

In [ ]:
if 'dbscan_df' not in locals():
    print("🚨 ERROR: No se puede generar el reporte, 'dbscan_df' no existe.")
else:
    print("\n--- Tabla de Resultados Detallada de DBSCAN (Mejores 10) ---")
    # Mostrar las 10 mejores combinaciones por Silhouette
    dbscan_df_sorted = dbscan_df.sort_values(by=['Silhouette Score', 'Num Clusters (valid)'], ascending=[False, False]).head(10)
    display(dbscan_df_sorted)

    # Guardar resultados completos a CSV
    results_csv_path = os.path.join(RESULTS_GRAPH_DIR, 'dbscan_parameter_search_results_extended.csv')
    dbscan_df.to_csv(results_csv_path, index=False)
    print(f"\nResultados detallados guardados en '{results_csv_path}'")

    # --- Visualización de Resultados (Mapas de Calor) ---
    print("\nGenerando mapas de calor de resultados para DBSCAN...")

    # Filtrar resultados donde se puedan calcular métricas (al menos 2 clusters válidos)
    plot_df = dbscan_df[dbscan_df['Num Clusters (valid)'] >= 2].copy()

    if not plot_df.empty:
        # Definición de función de ploteo para evitar repetición
        def plot_heatmap(df, value_col, title, cmap, fmt, filename):
            plt.figure(figsize=(14, 8))
            pivot_table = df.pivot_table(index='min_samples', columns='eps', values=value_col)
            sns.heatmap(pivot_table, annot=True, cmap=cmap, fmt=fmt, linewidths=.5, cbar_kws={'label': value_col})
            plt.title(f'DBSCAN: {title} por (Eps, Min Samples)', fontsize=16)
            plt.xlabel('Eps (Distancia Coseno)', fontsize=12)
            plt.ylabel('Min Samples', fontsize=12)
            plt.tight_layout()
            plt.savefig(os.path.join(RESULTS_GRAPH_DIR, filename), dpi=300)
            plt.close()

        # 1. Silhouette Score (Maximizar)
        plot_heatmap(plot_df, 'Silhouette Score', 'Silhouette Score (MAXIMIZAR)', 'viridis', ".2f", "dbscan_silhouette_heatmap.png")

        # 2. Davies-Bouldin Index (Minimizar)
        plot_heatmap(plot_df, 'Davies-Bouldin Index', 'Davies-Bouldin Index (MINIMIZAR)', 'cividis_r', ".2f", "dbscan_davies_bouldin_heatmap.png")

        # 3. Calinski-Harabasz Index (Maximizar)
        plot_heatmap(plot_df, 'Calinski-Harabasz Index', 'Calinski-Harabasz Index (MAXIMIZAR)', 'magma', ".0f", "dbscan_calinski_harabasz_heatmap.png")
        
        # 4. Número de Clusters Válidos
        plot_heatmap(dbscan_df, 'Num Clusters (valid)', 'Número de Clusters Válidos', 'Blues', "g", "dbscan_num_clusters_heatmap.png")

        # 5. Porcentaje de Ruido
        plot_heatmap(dbscan_df, 'Porcentaje Ruido', 'Porcentaje de Ruido', 'Reds', ".1f", "dbscan_noise_percentage_heatmap.png")
        
        print(f"✅ Mapas de calor guardados en: {RESULTS_GRAPH_DIR}")
        
    else:
        print("\n⚠️ Advertencia: No hay suficientes combinaciones de parámetros que resulten en al menos 2 clusters válidos para generar mapas de calor.")


    # --- Determinación de los Mejores Parámetros para DBSCAN ---
    print("\n--- Determinación de los Mejores Parámetros para DBSCAN ---")

    best_results_filtered = dbscan_df[dbscan_df['Num Clusters (valid)'] >= 2].copy()

    if not best_results_filtered.empty:
        # Mejor por Silhouette Score
        best_by_silhouette = best_results_filtered.loc[best_results_filtered['Silhouette Score'].idxmax()]
        
        # Mejor por Davies-Bouldin Index
        best_by_davies_bouldin = best_results_filtered.loc[best_results_filtered['Davies-Bouldin Index'].idxmin()]

        # Mejor por Calinski-Harabasz Index
        best_by_calinski_harabasz = best_results_filtered.loc[best_results_filtered['Calinski-Harabasz Index'].idxmax()]

        print(f"\nMejor por Silhouette (Eps={best_by_silhouette['eps']:.4f}, Min S={best_by_silhouette['min_samples']}): Score {best_by_silhouette['Silhouette Score']:.4f}")
        print(f"Mejor por Davies-Bouldin (Eps={best_by_davies_bouldin['eps']:.4f}, Min S={best_by_davies_bouldin['min_samples']}): Índice {best_by_davies_bouldin['Davies-Bouldin Index']:.4f}")
        print(f"Mejor por Calinski-Harabasz (Eps={best_by_calinski_harabasz['eps']:.4f}, Min S={best_by_calinski_harabasz['min_samples']}): Índice {best_by_calinski_harabasz['Calinski-Harabasz Index']:.0f}")

        # Sugerencia final basada en Silhouette
        optimal_dbscan_params_suggested = best_by_silhouette[['eps', 'min_samples']].to_dict()
        print(f"\n✨ DBSCAN | Parámetros sugeridos para usar: Eps={optimal_dbscan_params_suggested['eps']:.4f}, Min Samples={optimal_dbscan_params_suggested['min_samples']}")
    else:
        print("\nNo se encontraron combinaciones de parámetros con al menos 2 clusters válidos.")

    print("\n✔️ Análisis de optimización de DBSCAN completado.")

In [ ]:

if 'dbscan_df' not in locals() or 'embeddings_2d' not in locals():
    print("🚨 ERROR: Datos de DBSCAN o UMAP no disponibles.")
else:
    # --- Obtención de Parámetros Óptimos ---
    print("\n--- Determinación de los Mejores Parámetros para DBSCAN ---")
    best_results_filtered = dbscan_df[dbscan_df['Num Clusters (valid)'] >= 2].copy()

    if not best_results_filtered.empty:
        # Mejor por Silhouette Score
        best_by_silhouette = best_results_filtered.loc[best_results_filtered['Silhouette Score'].idxmax()]
        
        # Parámetros sugeridos para UMAP
        optimal_dbscan_params_suggested = best_by_silhouette[['eps', 'min_samples']].to_dict()
        opt_eps = optimal_dbscan_params_suggested['eps']
        opt_min_s = int(optimal_dbscan_params_suggested['min_samples'])
        print(f"\n✨ DBSCAN | Parámetros sugeridos (Silhouette): Eps={opt_eps:.4f}, Min Samples={opt_min_s}")
        
        # --- Generación del Clustering Óptimo ---
        dbscan_opt = DBSCAN(eps=opt_eps, min_samples=opt_min_s, metric='cosine')
        dbscan_labels = dbscan_opt.fit_predict(X)
        
        n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
        n_noise = list(dbscan_labels).count(-1)
        
        print(f"Resultado del Clustering óptimo: {n_clusters} clusters, {n_noise} puntos de ruido.")

        # --- Visualización UMAP de DBSCAN ---
        plt.figure(figsize=(12, 10))
        
        # El ruido (etiqueta -1) se maneja aparte
        unique_labels = np.unique(dbscan_labels)
        core_labels = unique_labels[unique_labels != -1]
        
        # Mapeo de colores: un color por cluster + negro o gris para ruido
        cmap_clusters = sns.color_palette("hsv", n_clusters)
        color_map = {label: cmap_clusters[i] for i, label in enumerate(core_labels)}
        color_map[-1] = 'gray' # Color para ruido
        
        # Asignar colores a cada punto
        point_colors = [color_map[label] for label in dbscan_labels]
        
        # Plotear puntos
        sns.scatterplot(
            x=embeddings_2d[:, 0], 
            y=embeddings_2d[:, 1], 
            hue=dbscan_labels, 
            palette=color_map, # Usamos el mapa de colores definido
            style=[1 if label == -1 else 0 for label in dbscan_labels], # Estilo diferente para ruido
            alpha=0.7,
            s=20
        )
        
        plt.title(f'DBSCAN (Eps={opt_eps:.3f}, Min S={opt_min_s}) sobre Embeddings UMAP', fontsize=16)
        plt.xlabel('UMAP 1', fontsize=12)
        plt.ylabel('UMAP 2', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.legend(title='Cluster ID', loc='best', fontsize='small')

        # Guardar gráfico
        umap_dbscan_path = os.path.join(RESULTS_GRAPH_DIR, f"umap_dbscan_eps_{opt_eps:.3f}_minS_{opt_min_s}_plot.png")
        plt.savefig(umap_dbscan_path, dpi=300)
        print(f"Gráfico DBSCAN/UMAP guardado en: {umap_dbscan_path}")
        plt.show()

    else:
        print("\nNo se encontraron combinaciones de parámetros con al menos 2 clusters válidos para visualización.")

    # ... (Resto del código de mapas de calor, si deseas mantenerlo en esta celda)
    print("\n✔️ Análisis de optimización de DBSCAN completado.")

## HDBSCAN

In [ ]:
if 'X' not in locals():
    print("🚨 ERROR: No se puede ejecutar HDBSCAN, 'X' (embeddings) no está cargado. Ejecute la Celda 1.")
else:

    print("\n--- BÚSQUEDA DE PARÁMETROS PARA HDBSCAN (METRIC='EUCLIDEAN' sobre L2 normalizados) ---")
    
    # --- Definición de Rangos de Búsqueda ---
    max_min_cluster_size = min(200, X.shape[0] // 10)
    
    if X.shape[0] < 5: 
        min_cluster_size_range = np.arange(2, X.shape[0] + 1)
    else:
        min_cluster_size_range = np.unique(np.sort(np.concatenate((
            np.arange(2, 20, 2),    
            np.arange(20, 101, 10),  
            np.arange(100, max_min_cluster_size + 1, 50)
        )))).astype(int) # Asegurar que sea entero
        
    min_samples_range = np.unique(np.sort(np.concatenate((
        np.arange(1, 10, 1),   
        np.arange(10, 50, 5),  
        np.arange(50, 101, 25) 
    ))))
    min_samples_range = min_samples_range[min_samples_range < X.shape[0]].astype(int) # Asegurar entero
    
    # Usar el valor por defecto
    cluster_selection_epsilon_range = [0.0] 
    
    # Rango para alpha (suavidad del modelo de densidad)
    alpha_range = np.unique(np.sort(np.array([0.5, 1.0, 1.5, 2.0])))
    
    # Imprimir rangos
    print(f"Rango de min_cluster_size: {min_cluster_size_range.min()}-{min_cluster_size_range.max()}")
    print(f"Rango de min_samples: {min_samples_range.min()}-{min_samples_range.max()}")
    print(f"Rango de alpha: {alpha_range}")

    # --- Loop de Búsqueda ---
    hdbscan_results = []
    
    param_combinations = list(itertools.product(
        min_cluster_size_range, 
        min_samples_range, 
        cluster_selection_epsilon_range, 
        alpha_range
    ))
    
    total_combinations = len(param_combinations)
    print(f"Combinaciones totales a probar: {total_combinations}")
    
    start_time = time.time()

    for min_size_val, min_samples_val, cluster_eps_val, alpha_val in tqdm(param_combinations, desc="Buscando HDBSCAN"):
        
        # Filtro: min_samples no debe ser mayor que min_cluster_size (aunque HDBSCAN lo ajusta)
        if min_samples_val > min_size_val:
            continue
            
        clusterer = hdbscan.HDBSCAN(min_cluster_size=min_size_val,
                                    min_samples=min_samples_val,
                                    cluster_selection_epsilon=cluster_eps_val,
                                    alpha=alpha_val,
                                    metric='euclidean', 
                                    prediction_data=False) # No necesitamos prediction_data para esta etapa
        labels = clusterer.fit_predict(X)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        total_points = len(labels)
        noise_percentage = (n_noise / total_points) * 100 if total_points > 0 else 0

        silhouette, davies_bouldin, calinski_harabasz = np.nan, np.nan, np.nan

        if n_clusters >= 2:
            labels_filtered = labels[labels != -1]
            data_filtered = X[labels != -1]

            if len(set(labels_filtered)) >= 2 and len(labels_filtered) >= 2:
                # Usar métricas estándar (Silhouette en euclidiana para ser coherente con la métrica de clustering)
                silhouette = silhouette_score(data_filtered, labels_filtered)
                davies_bouldin = davies_bouldin_score(data_filtered, labels_filtered)
                calinski_harabasz = calinski_harabasz_score(data_filtered, labels_filtered)

        hdbscan_results.append({
            'min_cluster_size': min_size_val,
            'min_samples': min_samples_val,
            'cluster_selection_epsilon': cluster_eps_val,
            'alpha': alpha_val,
            'Num Clusters (valid)': n_clusters,
            'Num Ruido': n_noise,
            'Porcentaje Ruido': noise_percentage,
            'Silhouette Score': silhouette,
            'Davies-Bouldin Index': davies_bouldin,
            'Calinski-Harabasz Index': calinski_harabasz
        })

    end_time = time.time()
    hdbscan_df = pd.DataFrame(hdbscan_results)
    
    print(f"\nOptimización HDBSCAN completa en {end_time - start_time:.2f} segundos.")
    print("✔️ Búsqueda de HDBSCAN completada.")

In [ ]:
if 'hdbscan_df' not in locals():
    print("🚨 ERROR: No se puede generar el reporte, 'hdbscan_df' no existe. Ejecute la Celda 8.")
elif 'embeddings_2d' not in locals():
    print("🚨 ERROR: 'embeddings_2d' (UMAP) no está disponible. Ejecute la Celda 4.")
else:
    # --- 1. Reporte y Guardado ---
    print("\n--- Tabla de Resultados Detallada de HDBSCAN (Mejores 10) ---")
    hdbscan_df_sorted = hdbscan_df.sort_values(by=['Silhouette Score', 'Num Clusters (valid)'], ascending=[False, False])
    display(hdbscan_df_sorted.head(10))

    results_csv_path = os.path.join(RESULTS_GRAPH_DIR, 'hdbscan_parameter_search_results_exhaustive.csv')
    hdbscan_df.to_csv(results_csv_path, index=False)
    print(f"\nResultados detallados guardados en '{results_csv_path}'")

    # --- 2. Determinación de los Mejores Parámetros ---
    print("\n--- Determinación de los Mejores Parámetros para HDBSCAN ---")
    best_results_filtered = hdbscan_df[hdbscan_df['Num Clusters (valid)'] >= 2].copy()

    if not best_results_filtered.empty:
        best_by_silhouette = best_results_filtered.loc[best_results_filtered['Silhouette Score'].idxmax()]
        
        # Parámetros sugeridos para UMAP
        optimal_hdbscan_params_suggested = best_by_silhouette[['min_cluster_size', 'min_samples', 'cluster_selection_epsilon', 'alpha']].to_dict()
        opt_mcs = int(optimal_hdbscan_params_suggested['min_cluster_size'])
        opt_ms = int(optimal_hdbscan_params_suggested['min_samples'])
        opt_cse = optimal_hdbscan_params_suggested['cluster_selection_epsilon']
        opt_alpha = optimal_hdbscan_params_suggested['alpha']
        
        print(f"\n✨ HDBSCAN | Parámetros sugeridos (Silhouette): mcs={opt_mcs}, ms={opt_ms}, cse={opt_cse:.2f}, alpha={opt_alpha:.1f}")

        # --- 3. Visualización de Heatmaps (Fijando cse y alpha óptimos) ---
        print("\nGenerando heatmaps para la mejor combinación de cse y alpha...")
        
        filtered_plot_df = best_results_filtered[
            (best_results_filtered['cluster_selection_epsilon'] == opt_cse) &
            (best_results_filtered['alpha'] == opt_alpha)
        ].copy()

        # Función de ploteo para evitar repetición
        def plot_heatmap(df, value_col, title, cmap, fmt, filename):
            plt.figure(figsize=(12, 8))
            # Usar min_cluster_size como columnas y min_samples como índice
            pivot_table = df.pivot_table(index='min_samples', columns='min_cluster_size', values=value_col)
            sns.heatmap(pivot_table, annot=True, cmap=cmap, fmt=fmt, linewidths=.5)
            plt.title(f'HDBSCAN: {title} (cse={opt_cse:.2f}, alpha={opt_alpha:.1f})', fontsize=16)
            plt.xlabel('Min Cluster Size', fontsize=12)
            plt.ylabel('Min Samples', fontsize=12)
            plt.tight_layout()
            plt.savefig(os.path.join(RESULTS_GRAPH_DIR, filename), dpi=300)
            plt.close()

        # Generar heatmaps
        if not filtered_plot_df.empty:
            plot_heatmap(filtered_plot_df, 'Silhouette Score', 'Silhouette Score (MAX)', 'viridis', ".2f", "hdbscan_silhouette_best_heatmap.png")
            plot_heatmap(filtered_plot_df, 'Porcentaje Ruido', 'Porcentaje de Ruido', 'Reds', ".1f", "hdbscan_noise_best_heatmap.png")
        
        # --- 4. Ejecución Óptima y Visualización UMAP ---
        clusterer_opt = hdbscan.HDBSCAN(min_cluster_size=opt_mcs,
                                            min_samples=opt_ms,
                                            cluster_selection_epsilon=opt_cse,
                                            alpha=opt_alpha,
                                            metric='euclidean',
                                            prediction_data=False)
        hdbscan_labels = clusterer_opt.fit_predict(X) 

        n_clusters = len(set(hdbscan_labels)) - (1 if -1 in hdbscan_labels else 0)
        n_noise = list(hdbscan_labels).count(-1)
        
        print(f"Resultado del Clustering óptimo: {n_clusters} clusters, {n_noise} puntos de ruido.")

        plt.figure(figsize=(12, 10))
        
        unique_labels = np.unique(hdbscan_labels)
        core_labels = unique_labels[unique_labels != -1]
        
        # Mapeo de colores: un color por cluster + gris para ruido
        cmap_clusters = sns.color_palette("hsv", n_clusters)
        color_map = {label: cmap_clusters[i] for i, label in enumerate(core_labels)}
        color_map[-1] = 'gray' 
        
        sns.scatterplot(
            x=embeddings_2d[:, 0], 
            y=embeddings_2d[:, 1], 
            hue=hdbscan_labels, 
            palette=color_map, 
            style=[1 if label == -1 else 0 for label in hdbscan_labels], 
            alpha=0.7,
            s=20
        )
        
        plt.title(f'HDBSCAN Óptimo (mcs={opt_mcs}, ms={opt_ms}) sobre Embeddings UMAP', fontsize=16)
        plt.xlabel('UMAP 1', fontsize=12)
        plt.ylabel('UMAP 2', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.legend(title='Cluster ID', loc='best', fontsize='small')

        umap_hdbscan_path = os.path.join(RESULTS_GRAPH_DIR, "umap_hdbscan_optimal_plot.png")
        plt.savefig(umap_hdbscan_path, dpi=300)
        print(f"Gráfico HDBSCAN/UMAP guardado en: {umap_hdbscan_path}")
        plt.show()

    else:
        print("\nNo se encontraron combinaciones de parámetros con al menos 2 clusters válidos.")

    print("\n✔️ Análisis de optimización de HDBSCAN completado.")

## Carga parámetros óptimos HDBSCAN

In [ ]:
# --- Configuración de Rutas ---
FINAL_CLUSTER_RESULTS_DIR = os.path.join(BASE_OUTPUT_DIR, "final_hdbscan_clusters")
os.makedirs(FINAL_CLUSTER_RESULTS_DIR, exist_ok=True)
print(f"Directorio de resultados final: {FINAL_CLUSTER_RESULTS_DIR}")

# --- Carga de Embeddings Normalizados ---
print(f"\nCargando embeddings normalizados desde: {EMBEDDINGS_PATH}")
try:
    df_embeddings = pd.read_csv(EMBEDDINGS_PATH)
    # Asumimos que la primera columna es ID y el resto son embeddings
    protein_ids = df_embeddings.iloc[:, 0]
    X = df_embeddings.iloc[:, 1:].values
    print("Embeddings normalizados cargados exitosamente.")
    
    # Verificación final de normalización L2 (y normalización si es necesario)
    norms = np.linalg.norm(X, axis=1)
    if not np.allclose(norms, 1.0, atol=1e-6):
        print("Advertencia: Normalizando L2 los embeddings.")
        X = X / norms[:, np.newaxis]
    else:
        print("Los embeddings están normalizados a L2=1.0. ¡Perfecto para euclidiana/coseno!")

except FileNotFoundError:
    print(f"🚨 ERROR: No se encontró el archivo de embeddings en la ruta: {EMBEDDINGS_PATH}. Usando datos de ejemplo.")
    np.random.seed(42)
    X = np.random.rand(500, 768) 
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
    protein_ids = pd.Series([f"Protein_{i}" for i in range(X.shape[0])], name='protein_id')
except Exception as e:
    print(f"🚨 ERROR: Ocurrió un error al cargar los embeddings: {e}. Usando datos de ejemplo.")
    np.random.seed(42)
    X = np.random.rand(500, 768) 
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
    protein_ids = pd.Series([f"Protein_{i}" for i in range(X.shape[0])], name='protein_id')

total_points = X.shape[0]

if total_points < 2:
    print("ERROR: No hay suficientes muestras en los embeddings para realizar clustering.")
    sys.exit()

# --- Definir los Parámetros Óptimos (AJUSTAR ESTOS VALORES) ---
# **IMPORTANTE**: Estos valores deben reflejar los resultados de tu Celda anterior.
# Usamos valores de ejemplo aquí.
OPTIMAL_MIN_CLUSTER_SIZE = 20 
OPTIMAL_MIN_SAMPLES = 20
OPTIMAL_CLUSTER_SELECTION_EPSILON = 0.0
OPTIMAL_ALPHA = 2.0 

print(f"\n--- Ejecutando HDBSCAN con parámetros óptimos ---")
print(f"  min_cluster_size = {OPTIMAL_MIN_CLUSTER_SIZE}")
print(f"  min_samples = {OPTIMAL_MIN_SAMPLES}")
print(f"  cluster_selection_epsilon = {OPTIMAL_CLUSTER_SELECTION_EPSILON:.2f}")
print(f"  alpha = {OPTIMAL_ALPHA:.1f}")

# --- Aplicar HDBSCAN ---
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=OPTIMAL_MIN_CLUSTER_SIZE,
    min_samples=OPTIMAL_MIN_SAMPLES,
    cluster_selection_epsilon=OPTIMAL_CLUSTER_SELECTION_EPSILON,
    alpha=OPTIMAL_ALPHA,
    metric='euclidean', # Correcto para L2 normalizados
    cluster_selection_method='eom',
    prediction_data=True
)

print("Entrenando y prediciendo clústeres con HDBSCAN...")
labels = clusterer.fit_predict(X)

# --- Análisis de Resultados y Métricas ---
n_clusters = len(np.unique(labels)) - (1 if -1 in labels else 0) 
n_noise = list(labels).count(-1)
noise_percentage = (n_noise / total_points) * 100 if total_points > 0 else 0

print(f"\n--- Resultados de HDBSCAN Final ---")
print(f"Número de clústeres válidos encontrados: {n_clusters}")
print(f"Número de puntos clasificados como ruido: {n_noise} ({noise_percentage:.2f}%)")

silhouette, davies_bouldin, calinski_harabasz = np.nan, np.nan, np.nan

if n_clusters >= 2:
    labels_filtered = labels[labels != -1]
    data_filtered = X[labels != -1]

    if len(set(labels_filtered)) >= 2 and len(labels_filtered) >= 2:
        # Usamos metric='cosine' para Silhouette, que es más fiel a la relación entre embeddings
        silhouette = silhouette_score(data_filtered, labels_filtered, metric='cosine') 
        # DB y CH usan métricas euclidianas por defecto (estándar de sklearn)
        davies_bouldin = davies_bouldin_score(data_filtered, labels_filtered)
        calinski_harabasz = calinski_harabasz_score(data_filtered, labels_filtered)
    else:
        print("Advertencia: Insuficientes clústeres válidos o puntos (después de filtrar ruido) para calcular métricas.")

print(f"Silhouette Score (coseno, excluyendo ruido): {silhouette:.4f}")
print(f"Davies-Bouldin Index (euclidiana, excluyendo ruido): {davies_bouldin:.4f}")
print(f"Calinski-Harabasz Index (euclidiana, excluyendo ruido): {calinski_harabasz:.4f}")

In [ ]:
if 'labels' not in locals():
    print("🚨 ERROR: Las etiquetas de HDBSCAN no están disponibles.")
else:
 
    
    # --- Guardar Resultados del Clustering ---

    # 1. Etiquetas de clúster por ID de proteína
    df_results = pd.DataFrame({
        'protein_id': protein_ids,
        'cluster_label': labels
    })
    
    # Usamos los parámetros en el nombre del archivo para trazabilidad
    params_suffix = f'mcs{OPTIMAL_MIN_CLUSTER_SIZE}_ms{OPTIMAL_MIN_SAMPLES}_cse{OPTIMAL_CLUSTER_SELECTION_EPSILON:.2f}_alpha{OPTIMAL_ALPHA:.1f}'
    output_labels_path = os.path.join(FINAL_CLUSTER_RESULTS_DIR, f'hdbscan_cluster_labels_{params_suffix}.csv')
    df_results.to_csv(output_labels_path, index=False)
    print(f"\n✅ Etiquetas de clúster guardadas en: {output_labels_path}")

    # 2. Resumen de clústeres
    df_cluster_summary = pd.DataFrame({
        'cluster_id': labels
    }).groupby('cluster_id').size().reset_index(name='size')
    
    # Excluir la fila de ruido para el resumen
    df_cluster_summary = df_cluster_summary[df_cluster_summary['cluster_id'] != -1].sort_values(by='size', ascending=False)
    
    output_summary_path = os.path.join(FINAL_CLUSTER_RESULTS_DIR, f'hdbscan_cluster_summary_{params_suffix}.csv')
    df_cluster_summary.to_csv(output_summary_path, index=False)
    print(f"✅ Resumen de clústeres guardado en: {output_summary_path}")

    # --- Visualización de Clústeres con UMAP ---
    print("\nGenerando Visualización UMAP para Clústeres Finales...")

    # Reducción de Dimensionalidad 
    try:
        # Si embeddings_2d existe, lo usamos
        if 'embeddings_2d' not in locals():
            raise NameError
        X_reduced = embeddings_2d
        print("Usando embeddings 2D (UMAP) pre-calculados.")
    except NameError:
        # Si no existe, lo calculamos
        reducer = umap.UMAP(n_components=2, random_state=42, metric='cosine', n_jobs=-1)
        X_reduced = reducer.fit_transform(X)
        print("Embeddings 2D (UMAP) calculados ahora.")

    plt.figure(figsize=(12, 10))

    # --- Lógica de Coloreado ---
    unique_cluster_labels = np.unique(labels)
    n_clusters = len(unique_cluster_labels) - (1 if -1 in unique_cluster_labels else 0)
    
    # Color para el ruido
    noise_color = 'lightgray'

    # Paleta de colores para los clústeres
    if n_clusters > 0:
        # Usamos una paleta de seaborn para obtener colores diferenciados
        cluster_colors = sns.color_palette("hsv", n_clusters)
        color_map = {}
        cluster_id_counter = 0
        for label in sorted(unique_cluster_labels):
            if label == -1:
                color_map[label] = noise_color
            else:
                color_map[label] = cluster_colors[cluster_id_counter]
                cluster_id_counter += 1
    else:
        # Si solo hay ruido, solo coloreamos el ruido
        color_map = {-1: noise_color}


    # Plotear usando Scatterplot (más simple y maneja mejor la leyenda)
    plot_data = pd.DataFrame(X_reduced, columns=['UMAP_1', 'UMAP_2'])
    plot_data['Cluster'] = labels
    
    # Ordenar para que los puntos de ruido queden abajo y los clústeres en la parte superior
    plot_data = plot_data.sort_values(by='Cluster', ascending=True)

    sns.scatterplot(
        x='UMAP_1', 
        y='UMAP_2', 
        hue='Cluster', 
        palette=color_map, 
        s=15, 
        alpha=0.7,
        data=plot_data
    )

    plt.title(f'HDBSCAN Clusters Final (mcs={OPTIMAL_MIN_CLUSTER_SIZE}, ms={OPTIMAL_MIN_SAMPLES}) - UMAP 2D', fontsize=16)
    plt.xlabel('UMAP Component 1', fontsize=12)
    plt.ylabel('UMAP Component 2', fontsize=12)
    plt.legend(title='Cluster ID', bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout(rect=[0, 0, 0.85, 1]) 

    output_umap_plot_path = os.path.join(FINAL_CLUSTER_RESULTS_DIR, f'hdbscan_umap_clusters_final_{params_suffix}.png')
    plt.savefig(output_umap_plot_path, dpi=300)
    print(f"✅ Visualización UMAP guardada en: {output_umap_plot_path}")
    plt.show()

    print("\n--- Proceso de Clustering HDBSCAN Final Completado ---")